In [3]:
from pathlib import Path
import pickle
import pandas as pd
import numpy as np
import lightgbm as lgb
from scipy.sparse import load_npz

p = Path(".")

train = pd.read_csv(p / "reranker-train.txt", index_col=0)
test = pd.read_csv(p / "reranker-test.txt", index_col=0)
ent = pd.read_csv(p / "users-entropy.txt", index_col=0)

with open(p / "mappings_item_filters_user_filters.pkl", "rb") as f:
    maps = pickle.load(f)

with open(p / "test_true_item_filters_user_filters.pkl", "rb") as f:
    test_true = pickle.load(f)

with open(p / "reranker_feature_cols.pkl", "rb") as f:
    cols = pickle.load(f)

mat = load_npz(p / "user_item_matrix_item_filters_user_filters.npz")
ranker = lgb.Booster(model_file=str(p / "lightgbm_reranker.txt"))

In [4]:
uc = [
    "user_n_events", "n_unique_items", "user_n_likes", "n_listens",
    "user_like_rate", "user_avg_played_ratio", "user_organic_ratio",
    "entropy", "entropy_norm", "n_items", "matrix_n_positive",
    "matrix_total_weight"
]

ic = [
    "item_n_events", "n_unique_users", "item_n_likes", "item_like_rate",
    "item_avg_played_ratio", "avg_track_length", "item_organic_ratio",
    "matrix_popularity"
]

vc = [
    "ui_n_events", "mean_played_ratio", "has_like", "days_before_test",
    "organic_ratio_ui", "interaction_weight"
]

uc = [c for c in uc if c in cols]
ic = [c for c in ic if c in cols]
vc = [c for c in vc if c in cols]

### Аудит данных
Смотрим размер train/test, количество пользователей, айтемов и структуру признаков reranker.

In [5]:
test["base_score"] = ranker.predict(test[cols])

aud = pd.DataFrame({
    "block": [
        "train rows",
        "test rows",
        "test users",
        "test items",
        "matrix shape",
        "feature count",
        "user features",
        "item features",
        "user-item features"
    ],
    "value": [
        train.shape[0],
        test.shape[0],
        test["uid"].nunique(),
        test["item_id"].nunique(),
        str(mat.shape),
        len(cols),
        len(uc),
        len(ic),
        len(vc)
    ]
})

aud

,block,value
0,train rows,354300
1,test rows,88600
2,test users,886
3,test items,4997
4,matrix shape,"(6477, 200250)"
5,feature count,27
6,user features,12
7,item features,8
8,user-item features,6


In [6]:
def met(d, k=10):
    x = d.sort_values("base_score", ascending=False).head(k)
    y = x["label"].values
    rel = d["label"].sum()
    rec = y.sum() / max(rel, 1)
    hit = int(y.sum() > 0)
    dcg = sum(y[i] / np.log2(i + 2) for i in range(len(y)))
    best = np.sort(d["label"].values)[::-1][:k]
    idcg = sum(best[i] / np.log2(i + 2) for i in range(len(best)))
    ndcg = dcg / idcg if idcg > 0 else 0
    return pd.Series({
        "cand": len(d),
        "rel_cand": rel,
        "top10_rel": y.sum(),
        "recall": rec,
        "ndcg": ndcg,
        "hit": hit,
        "avg_score_top10": x["base_score"].mean()
    })

usr = test.groupby("uid").apply(met).reset_index()

usr = usr.merge(
    test.groupby("uid")[[
        "user_n_events", "n_unique_items", "user_n_likes",
        "n_listens", "user_like_rate", "entropy_norm",
        "matrix_total_weight"
    ]].first().reset_index(),
    on="uid",
    how="left"
)

usr = usr[usr["rel_cand"] > 0].copy()

usr["act_group"] = pd.qcut(usr["user_n_events"], 3, labels=["low_activity", "mid_activity", "high_activity"])
usr["ent_group"] = pd.qcut(usr["entropy_norm"], 3, labels=["low_entropy", "mid_entropy", "high_entropy"])

usr.head()

C:\Users\vedma\AppData\Local\Temp\ipykernel_9052\3519727273.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  usr = test.groupby("uid").apply(met).reset_index()


,uid,cand,rel_cand,top10_rel,recall,ndcg,hit,avg_score_top10,user_n_events,n_unique_items,user_n_likes,n_listens,user_like_rate,entropy_norm,matrix_total_weight,act_group,ent_group
1,2200,100.0,1.0,0.0,0.0,0.000000,0.0,0.122978,1299,823,32,1267,0.024634,0.968864,734.0,mid_activity,high_entropy
2,4400,100.0,3.0,0.0,0.0,0.000000,0.0,-0.006930,345,296,0,345,0.000000,0.972341,216.0,low_activity,high_entropy
4,7800,100.0,4.0,0.0,0.0,0.000000,0.0,-0.002861,1531,1235,26,1505,0.016982,0.989093,1124.0,mid_activity,high_entropy
5,9900,100.0,3.0,0.0,0.0,0.000000,0.0,0.081436,982,699,1,981,0.001018,0.986958,537.0,low_activity,high_entropy
6,10300,100.0,2.0,1.0,0.5,0.218407,1.0,0.059319,826,621,0,826,0.000000,0.990078,591.0,low_activity,high_entropy


### Выбор пользователей для стресс-теста
Выбираем разных пользователей по активности и энтропии, чтобы проверить модель не только на одном типе поведения.

In [7]:
pick = []

for a in ["low_activity", "mid_activity", "high_activity"]:
    for e in ["low_entropy", "mid_entropy", "high_entropy"]:
        x = usr[(usr["act_group"] == a) & (usr["ent_group"] == e)]
        if len(x) > 0:
            pick.append(x.sort_values(["rel_cand", "ndcg"], ascending=[False, True]).head(2))

sel = pd.concat(pick).drop_duplicates("uid").head(18)

sel[[
    "uid", "act_group", "ent_group", "cand", "rel_cand", "top10_rel",
    "recall", "ndcg", "hit", "user_n_events", "n_unique_items",
    "user_like_rate", "entropy_norm", "matrix_total_weight"
]].sort_values(["act_group", "ent_group", "uid"])

,uid,act_group,ent_group,cand,rel_cand,top10_rel,recall,ndcg,hit,user_n_events,n_unique_items,user_like_rate,entropy_norm,matrix_total_weight
12,14700,low_activity,low_entropy,100.0,10.0,4.0,0.400000,0.563788,1.0,483,169,0.000000,0.798679,59.0
235,268700,low_activity,low_entropy,100.0,13.0,9.0,0.692308,0.936379,1.0,860,198,0.002326,0.934644,53.0
456,509100,low_activity,mid_entropy,100.0,12.0,0.0,0.000000,0.000000,0.0,711,429,0.000000,0.963623,96.0
708,790600,low_activity,mid_entropy,100.0,6.0,1.0,0.166667,0.095460,1.0,690,366,0.114493,0.963046,144.0
768,865000,low_activity,high_entropy,100.0,12.0,0.0,0.000000,0.000000,0.0,748,534,0.020053,0.972238,198.0
772,870000,low_activity,high_entropy,100.0,13.0,5.0,0.384615,0.485814,1.0,540,331,0.107407,0.969351,103.0
219,247700,mid_activity,low_entropy,100.0,10.0,3.0,0.300000,0.312049,1.0,2339,745,0.005558,0.907958,106.0
694,774700,mid_activity,low_entropy,100.0,17.0,5.0,0.294118,0.605505,1.0,2106,577,0.000950,0.909225,110.0
55,58300,mid_activity,mid_entropy,100.0,14.0,2.0,0.142857,0.289523,1.0,2327,1225,0.006016,0.961728,573.0
319,358600,mid_activity,mid_entropy,100.0,11.0,6.0,0.545455,0.696840,1.0,1804,874,0.000554,0.951249,619.0


In [9]:
uids = sel["uid"].tolist()

def topx(d, col="base_score", k=10):
    return d.sort_values(col, ascending=False).head(k)

def calc(d, b, uid, s):
    x = d.copy()
    x["score"] = ranker.predict(x[cols])
    y = topx(x, "score", 10)
    base = topx(b, "score", 10)

    true = len(test_true.get(uid, []))
    rel_pool = x["label"].sum()
    rel_base_pool = b["label"].sum()

    return {
        "uid": uid,
        "scenario": s,
        "cand": len(x),
        "true_items": true,
        "rel_pool": rel_pool,
        "base_rel_pool": rel_base_pool,
        "lost_rel_pool": rel_base_pool - rel_pool,
        "pool_recall": rel_pool / max(true, 1),
        "top10_rel": y["label"].sum(),
        "base_top10_rel": base["label"].sum(),
        "delta_top10_rel": y["label"].sum() - base["label"].sum(),
        "hit": int(y["label"].sum() > 0),
        "overlap_top10": len(set(y["item_id"]) & set(base["item_id"])) / max(len(set(y["item_id"]) | set(base["item_id"])), 1),
        "changed_items": 10 - len(set(y["item_id"]) & set(base["item_id"]))
    }

def br1(d, s):
    x = d.copy()
    if s == "base":
        return x
    if s == "drop_top10":
        return x[x["elsa_rank"] >= 10].copy()
    if s == "drop_top20":
        return x[x["elsa_rank"] >= 20].copy()
    if s == "only_top50":
        return x.sort_values("elsa_rank").head(50).copy()
    if s == "only_tail50":
        return x.sort_values("elsa_rank", ascending=False).head(50).copy()
    if s == "popular_bias50":
        return x.sort_values("matrix_popularity", ascending=False).head(50).copy()
    if s == "random50":
        return x.sample(50, random_state=42).copy()
    return x

In [10]:
sc1 = [
    "base",
    "drop_top10",
    "drop_top20",
    "only_top50",
    "only_tail50",
    "popular_bias50",
    "random50"
]

r1 = []

for uid in uids:
    d = test[test["uid"] == uid].copy()
    d["score"] = ranker.predict(d[cols])
    b = br1(d, "base")

    for s in sc1:
        x = br1(d, s)
        r1.append(calc(x, b, uid, s))

s1 = pd.DataFrame(r1)
s1.head()

,uid,scenario,cand,true_items,rel_pool,base_rel_pool,lost_rel_pool,pool_recall,top10_rel,base_top10_rel,delta_top10_rel,hit,overlap_top10,changed_items
0,268700,base,100,97,13,13,0,0.134021,9,9,0,1,1.000000,0
1,268700,drop_top10,90,97,12,13,1,0.123711,10,9,1,1,0.818182,1
2,268700,drop_top20,80,97,11,13,2,0.113402,10,9,1,1,0.666667,2
3,268700,only_top50,50,97,8,13,5,0.082474,8,9,-1,1,0.333333,5
4,268700,only_tail50,50,97,5,13,8,0.051546,5,9,-4,1,0.333333,5


### Стресс-тест первого этапа
Сравниваем, как меняется candidate pool и top-10 при поломке candidate generation.

In [11]:
agg1 = (s1
    .groupby("scenario")
    .agg(
        users=("uid", "nunique"),
        avg_cand=("cand", "mean"),
        avg_rel_pool=("rel_pool", "mean"),
        avg_lost_rel_pool=("lost_rel_pool", "mean"),
        avg_pool_recall=("pool_recall", "mean"),
        avg_top10_rel=("top10_rel", "mean"),
        avg_delta_top10_rel=("delta_top10_rel", "mean"),
        hit_rate=("hit", "mean"),
        avg_overlap_top10=("overlap_top10", "mean"),
        avg_changed_items=("changed_items", "mean"))
    .reset_index()
    .sort_values("avg_overlap_top10"))

agg1

,scenario,users,avg_cand,avg_rel_pool,avg_lost_rel_pool,avg_pool_recall,avg_top10_rel,avg_delta_top10_rel,hit_rate,avg_overlap_top10,avg_changed_items
3,only_tail50,18,50.0,4.611111,7.555556,0.043935,1.333333,-1.611111,0.722222,0.209529,6.722222
6,random50,18,50.0,6.444444,5.722222,0.056045,2.500000,-0.444444,0.944444,0.345624,5.055556
2,drop_top20,18,80.0,8.500000,3.666667,0.078646,2.555556,-0.388889,0.833333,0.484917,3.722222
5,popular_bias50,18,50.0,6.611111,5.555556,0.061349,2.333333,-0.611111,0.833333,0.515983,3.333333
4,only_top50,18,50.0,7.555556,4.611111,0.068300,2.833333,-0.111111,0.944444,0.531393,3.277778
1,drop_top10,18,90.0,9.888889,2.277778,0.091867,2.666667,-0.277778,0.833333,0.629852,2.444444
0,base,18,100.0,12.166667,0.000000,0.112235,2.944444,0.000000,0.888889,1.000000,0.000000


### Чувствительность пользователей к первому этапу
Проверяем, у каких пользователей сильнее всего меняется выдача при деградации candidate generation.

In [13]:
u1 = (s1[s1["scenario"] != "base"]
    .groupby("uid")
    .agg(
        avg_lost_rel_pool=("lost_rel_pool", "mean"),
        max_lost_rel_pool=("lost_rel_pool", "max"),
        avg_delta_top10_rel=("delta_top10_rel", "mean"),
        min_delta_top10_rel=("delta_top10_rel", "min"),
        avg_overlap_top10=("overlap_top10", "mean"),
        max_changed_items=("changed_items", "max"))
    .reset_index())

u1 = u1.merge(
    sel[[
        "uid", "act_group", "ent_group", "rel_cand", "top10_rel",
        "recall", "ndcg", "user_n_events", "n_unique_items",
        "user_like_rate", "entropy_norm", "matrix_total_weight"
    ]],
    on="uid",
    how="left"
)

u1.sort_values(["avg_overlap_top10", "max_lost_rel_pool"], ascending=[True, False])

,uid,avg_lost_rel_pool,max_lost_rel_pool,avg_delta_top10_rel,min_delta_top10_rel,avg_overlap_top10,max_changed_items,act_group,ent_group,rel_cand,top10_rel,recall,ndcg,user_n_events,n_unique_items,user_like_rate,entropy_norm,matrix_total_weight
12,408800,3.000000,5,-0.833333,-1,0.315234,7,mid_activity,high_entropy,7.0,1.0,0.142857,0.079457,1547,1028,0.016160,0.969836,75.0
0,14700,4.500000,7,-2.333333,-3,0.335317,6,low_activity,low_entropy,10.0,4.0,0.400000,0.563788,483,169,0.000000,0.798679,59.0
14,774700,7.000000,9,-2.500000,-4,0.341581,8,mid_activity,low_entropy,17.0,5.0,0.294118,0.605505,2106,577,0.000950,0.909225,110.0
15,790600,3.000000,5,-0.666667,-1,0.373124,7,low_activity,mid_entropy,6.0,1.0,0.166667,0.095460,690,366,0.114493,0.963046,144.0
16,865000,5.000000,8,0.500000,0,0.401563,8,low_activity,high_entropy,12.0,0.0,0.000000,0.000000,748,534,0.020053,0.972238,198.0
13,509100,3.666667,7,1.166667,1,0.401709,5,low_activity,mid_entropy,12.0,0.0,0.000000,0.000000,711,429,0.000000,0.963623,96.0
3,152000,3.500000,6,-1.000000,-2,0.422212,8,mid_activity,high_entropy,7.0,2.0,0.285714,0.201129,2286,1447,0.010061,0.979585,1247.0
4,155300,4.333333,7,-1.833333,-3,0.425627,7,high_activity,low_entropy,12.0,6.0,0.500000,0.661831,2950,714,0.000339,0.874182,14.0
8,348200,3.666667,7,-0.666667,-2,0.433292,9,high_activity,high_entropy,7.0,2.0,0.285714,0.361590,4417,2461,0.000453,0.966029,1212.0
17,870000,5.000000,9,-0.833333,-4,0.435734,9,low_activity,high_entropy,13.0,5.0,0.384615,0.485814,540,331,0.107407,0.969351,103.0


### Худший сценарий первого этапа по пользователям
Для каждого пользователя выбираем сценарий, где потери в top-10 максимальны.

In [15]:
d1 = s1[s1["scenario"] != "base"].merge(
    sel[[
        "uid", "act_group", "ent_group", "rel_cand",
        "recall", "ndcg", "user_n_events", "n_unique_items",
        "user_like_rate", "entropy_norm", "matrix_total_weight"
    ]],
    on="uid",
    how="left"
)

w1 = (d1
    .sort_values(
        ["delta_top10_rel", "lost_rel_pool", "changed_items"],
        ascending=[True, False, False]
    )
    .groupby("uid")
    .head(1))

w1[[
    "uid", "scenario", "act_group", "ent_group",
    "base_rel_pool", "rel_pool", "lost_rel_pool",
    "base_top10_rel", "top10_rel", "delta_top10_rel",
    "overlap_top10", "changed_items",
    "user_n_events", "n_unique_items", "entropy_norm"
]].sort_values(["delta_top10_rel", "lost_rel_pool"])

,uid,scenario,act_group,ent_group,base_rel_pool,rel_pool,lost_rel_pool,base_top10_rel,top10_rel,delta_top10_rel,overlap_top10,changed_items,user_n_events,n_unique_items,entropy_norm
57,358600,only_tail50,mid_activity,mid_entropy,11,2,9,6,1,-5,0.052632,9,1804,874,0.951249
3,268700,only_tail50,low_activity,low_entropy,13,5,8,9,5,-4,0.333333,5,860,198,0.934644
27,870000,only_tail50,low_activity,high_entropy,13,4,9,5,1,-4,0.052632,9,540,331,0.969351
39,774700,only_tail50,mid_activity,low_entropy,17,8,9,5,1,-4,0.111111,8,2106,577,0.909225
9,14700,only_tail50,low_activity,low_entropy,10,3,7,4,1,-3,0.250000,6,483,169,0.798679
83,155300,random50,high_activity,low_entropy,12,5,7,6,3,-3,0.333333,5,2950,714,0.874182
69,152000,only_tail50,mid_activity,high_entropy,7,1,6,2,0,-2,0.111111,8,2286,1447,0.979585
105,348200,only_tail50,high_activity,high_entropy,7,0,7,2,0,-2,0.052632,9,4417,2461,0.966029
100,319200,popular_bias50,high_activity,high_entropy,20,6,14,3,1,-2,0.538462,3,3571,1806,0.967272
65,408800,random50,mid_activity,high_entropy,7,2,5,1,0,-1,0.176471,7,1547,1028,0.969836


### Поломка первого этапа по типам пользователей
Смотрим, какие группы пользователей сильнее страдают от потери или смещения кандидатов.

In [16]:
g1 = (d1
    .groupby(["act_group", "ent_group", "scenario"])
    .agg(
        users=("uid", "nunique"),
        lost_rel_pool=("lost_rel_pool", "mean"),
        delta_top10_rel=("delta_top10_rel", "mean"),
        overlap_top10=("overlap_top10", "mean"),
        changed_items=("changed_items", "mean")
    )
    .reset_index()
    .sort_values(["delta_top10_rel", "lost_rel_pool"], ascending=[True, False]))

g1.head(20)

C:\Users\vedma\AppData\Local\Temp\ipykernel_9052\2887492697.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["act_group", "ent_group", "scenario"])


,act_group,ent_group,scenario,users,lost_rel_pool,delta_top10_rel,overlap_top10,changed_items
2,low_activity,low_entropy,only_tail50,2,7.5,-3.5,0.291667,5.5
5,low_activity,low_entropy,random50,2,6.5,-3.0,0.333333,5.0
26,mid_activity,mid_entropy,only_tail50,2,8.5,-2.5,0.151316,7.5
50,high_activity,high_entropy,only_tail50,2,9.5,-2.0,0.192982,7.0
22,mid_activity,low_entropy,popular_bias50,2,6.5,-2.0,0.421569,4.5
20,mid_activity,low_entropy,only_tail50,2,6.0,-2.0,0.111111,8.0
52,high_activity,high_entropy,popular_bias50,2,9.0,-1.5,0.538462,3.0
14,low_activity,high_entropy,only_tail50,2,8.5,-1.5,0.081871,8.5
29,mid_activity,mid_entropy,random50,2,7.0,-1.5,0.602564,2.5
23,mid_activity,low_entropy,random50,2,6.5,-1.5,0.435897,4.0


### Пример изменения top-10 на первом этапе
Показываем конкретного пользователя: какие айтемы выпали из top-10 и какие пришли вместо них.

In [17]:
def show1(uid, s):
    d = test[test["uid"] == uid].copy()
    d["score"] = ranker.predict(d[cols])

    b = br1(d, "base").copy()
    x = br1(d, s).copy()

    b["score"] = ranker.predict(b[cols])
    x["score"] = ranker.predict(x[cols])

    bt = b.sort_values("score", ascending=False).head(10)[[
        "item_id", "label", "elsa_rank", "score",
        "matrix_popularity", "item_like_rate"
    ]].copy()

    xt = x.sort_values("score", ascending=False).head(10)[[
        "item_id", "label", "elsa_rank", "score",
        "matrix_popularity", "item_like_rate"
    ]].copy()

    bt["base_rank"] = range(1, len(bt) + 1)
    xt["broken_rank"] = range(1, len(xt) + 1)

    z = bt.merge(xt, on="item_id", how="outer", suffixes=("_base", "_broken"))

    z["status"] = np.where(
        z["base_rank"].notna() & z["broken_rank"].notna(),
        "kept",
        np.where(z["base_rank"].notna(), "dropped_from_top10", "new_in_top10")
    )

    return z.sort_values(["base_rank", "broken_rank"], na_position="last")

uid0 = int(w1.iloc[0]["uid"])
s0 = w1.iloc[0]["scenario"]

print(uid0, s0)
show1(uid0, s0)

358600 only_tail50


,item_id,label_base,elsa_rank_base,score_base,matrix_popularity_base,item_like_rate_base,base_rank,label_broken,elsa_rank_broken,score_broken,matrix_popularity_broken,item_like_rate_broken,broken_rank,status
13,6275528,1.0,39.0,0.331290,621.0,0.012544,1.0,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10
15,6484680,1.0,5.0,0.329711,449.0,0.009138,2.0,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10
0,644501,1.0,4.0,0.318197,484.0,0.006423,3.0,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10
9,4952290,1.0,20.0,0.288454,484.0,0.008654,4.0,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10
6,3588069,0.0,25.0,0.287537,101.0,0.006369,5.0,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10
4,2654961,0.0,18.0,0.259334,380.0,0.013713,6.0,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10
3,2118649,0.0,31.0,0.258277,628.0,0.014553,7.0,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10
1,966876,1.0,92.0,0.257824,160.0,0.014070,8.0,1.0,92.0,0.257824,160.0,0.014070,1.0,kept
12,6197362,0.0,16.0,0.251993,254.0,0.006985,9.0,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10
5,3061013,1.0,45.0,0.243519,667.0,0.014497,10.0,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10


In [32]:
def br2(d, s):
    x = d.copy()

    if s == "base":
        return x

    if s == "rank_nan":
        x["elsa_rank"] = np.nan

    if s == "rank_reverse":
        x["elsa_rank"] = x["elsa_rank"].max() - x["elsa_rank"]

    if s == "user_nan":
        x[uc] = np.nan

    if s == "user_low":
        x[uc] = x[uc] * 0.1

    if s == "user_high":
        x[uc] = x[uc] * 2

    if s == "item_nan":
        x[ic] = np.nan

    if s == "item_low":
        x[ic] = x[ic] * 0.1

    if s == "item_high":
        x[ic] = x[ic] * 2

    if s == "ui_nan":
        x[vc] = np.nan

    if s == "ui_default":
        for c in vc:
            x[c] = np.nan
        if "ui_n_events" in x.columns:
            x["ui_n_events"] = 0
        if "has_like" in x.columns:
            x["has_like"] = 0
        if "interaction_weight" in x.columns:
            x["interaction_weight"] = 0
        if "days_before_test" in x.columns:
            x["days_before_test"] = -1

    return x

In [34]:
def calc2(d, x, uid, s):
    b = d.copy()
    y = x.copy()

    b["score"] = ranker.predict(b[cols])
    y["score"] = ranker.predict(y[cols])

    bt = b.sort_values("score", ascending=False).head(10)
    yt = y.sort_values("score", ascending=False).head(10)

    bi = set(bt["item_id"])
    yi = set(yt["item_id"])

    br = b.sort_values("score", ascending=False)[["item_id", "label", "score"]].copy()
    yr = y.sort_values("score", ascending=False)[["item_id", "score"]].copy()

    br["base_rank"] = range(1, len(br) + 1)
    yr["broken_rank"] = range(1, len(yr) + 1)

    z = br.merge(yr, on="item_id", how="inner", suffixes=("_base", "_broken"))
    z["score_shift"] = z["score_broken"] - z["score_base"]
    z["rank_shift"] = z["broken_rank"] - z["base_rank"]

    rel_drop = len(set(bt[bt["label"] == 1]["item_id"]) - yi)

    return {
        "uid": uid,
        "scenario": s,
        "base_rel_pool": b["label"].sum(),
        "rel_pool": y["label"].sum(),
        "base_top10_rel": bt["label"].sum(),
        "top10_rel": yt["label"].sum(),
        "delta_top10_rel": yt["label"].sum() - bt["label"].sum(),
        "relevant_dropped_from_top10": rel_drop,
        "hit": int(yt["label"].sum() > 0),
        "overlap_top10": len(bi & yi) / max(len(bi | yi), 1),
        "changed_items": 10 - len(bi & yi),
        "mean_abs_score_shift": z["score_shift"].abs().mean(),
        "max_abs_score_shift": z["score_shift"].abs().max(),
        "mean_abs_rank_shift": z["rank_shift"].abs().mean(),
        "max_abs_rank_shift": z["rank_shift"].abs().max()
    }

### Стресс-тест второго этапа
Ломаем группы признаков reranker и смотрим, как меняется порядок внутри одного и того же candidate pool.

In [36]:
sc2 = [
    "base",
    "rank_nan",
    "rank_reverse",
    "user_nan",
    "user_low",
    "user_high",
    "item_nan",
    "item_low",
    "item_high",
    "ui_nan",
    "ui_default"
]

r2 = []

for uid in uids:
    d = test[test["uid"] == uid].copy()

    for s in sc2:
        x = br2(d, s)
        r2.append(calc2(d, x, uid, s))

s2 = pd.DataFrame(r2)

agg2 = (s2
    .groupby("scenario")
    .agg(
        users=("uid", "nunique"),
        avg_delta_top10_rel=("delta_top10_rel", "mean"),
        avg_relevant_dropped=("relevant_dropped_from_top10", "mean"),
        hit_rate=("hit", "mean"),
        avg_overlap_top10=("overlap_top10", "mean"),
        avg_changed_items=("changed_items", "mean"),
        avg_abs_score_shift=("mean_abs_score_shift", "mean"),
        max_abs_score_shift=("max_abs_score_shift", "mean"),
        avg_abs_rank_shift=("mean_abs_rank_shift", "mean"),
        max_abs_rank_shift=("max_abs_rank_shift", "mean"))
    .reset_index()
    .sort_values("avg_overlap_top10"))

agg2

,scenario,users,avg_delta_top10_rel,avg_relevant_dropped,hit_rate,avg_overlap_top10,avg_changed_items,avg_abs_score_shift,max_abs_score_shift,avg_abs_rank_shift,max_abs_rank_shift
6,ui_default,18,-1.111111,2.055556,0.833333,0.251696,6.333333,0.107132,0.814361,12.195556,75.944444
7,ui_nan,18,-1.111111,2.055556,0.833333,0.251696,6.333333,0.107132,0.814361,12.195556,75.944444
8,user_high,18,-0.388889,0.666667,0.833333,0.590746,2.888889,0.159735,0.353819,11.472222,47.277778
3,item_nan,18,-0.055556,0.555556,0.888889,0.598946,2.777778,0.074929,0.256538,13.022222,54.277778
9,user_low,18,-0.111111,0.555556,0.833333,0.600742,2.777778,0.192953,0.443176,11.654444,47.111111
2,item_low,18,0.055556,0.611111,0.833333,0.610401,2.722222,0.072828,0.246401,13.551111,56.666667
1,item_high,18,-0.055556,0.555556,0.833333,0.614338,2.666667,0.120125,0.348153,13.868889,51.277778
10,user_nan,18,0.055556,0.444444,0.888889,0.616948,2.611111,0.175398,0.420662,11.442222,47.500000
5,rank_reverse,18,0.000000,0.333333,0.888889,0.689066,2.055556,0.102530,0.387725,21.413333,67.777778
4,rank_nan,18,-0.388889,0.500000,0.833333,0.704974,2.000000,0.214923,0.463031,18.684444,65.388889


На втором этапе candidate pool не меняется, поэтому поломка влияет не на наличие кандидатов, а на их порядок. Самая критичная группа — user-item признаки (ui_nan, ui_default). При их поломке в среднем меняется 6.33 объекта из top-10, overlap с базовой выдачей падает до 0.25, а из исходного top-10 выпадает в среднем 2.06 релевантных объекта. Это сильнее, чем поломка user/item признаков и сильнее, чем поломка elsa_rank.

Бизнесово это означает: модель сильно зависит от признаков конкретной связи пользователь-трек. Если не пришли признаки вроде has_like, interaction_weight, mean_played_ratio, days_before_test, модель перестает понимать, насколько конкретный трек связан с конкретным пользователем, и начинает переставлять выдачу почти заново.

### Чувствительность пользователей ко второму этапу
Определяем пользователей, у которых поломка признаков reranker сильнее всего ухудшает top-10.

In [39]:
d2 = s2[s2["scenario"] != "base"].merge(
    sel[[
        "uid", "act_group", "ent_group", "rel_cand",
        "recall", "ndcg", "user_n_events", "n_unique_items",
        "user_like_rate", "entropy_norm", "matrix_total_weight"
    ]],
    on="uid",
    how="left"
)

w2 = (d2
    .sort_values(
        ["delta_top10_rel", "relevant_dropped_from_top10", "changed_items"],
        ascending=[True, False, False]
    )
    .groupby("uid")
    .head(1))

w2[[
    "uid", "scenario", "act_group", "ent_group",
    "base_top10_rel", "top10_rel", "delta_top10_rel",
    "relevant_dropped_from_top10",
    "overlap_top10", "changed_items",
    "mean_abs_score_shift", "mean_abs_rank_shift",
    "user_n_events", "n_unique_items", "entropy_norm"
]].sort_values(["delta_top10_rel", "relevant_dropped_from_top10"])

,uid,scenario,act_group,ent_group,base_top10_rel,top10_rel,delta_top10_rel,relevant_dropped_from_top10,overlap_top10,changed_items,mean_abs_score_shift,mean_abs_rank_shift,user_n_events,n_unique_items,entropy_norm
8,268700,ui_nan,low_activity,low_entropy,9,1,-8,9,0.052632,9,0.157828,9.44,860,198,0.934644
138,155300,ui_nan,high_activity,low_entropy,6,1,-5,5,0.176471,7,0.107959,12.34,2950,714,0.874182
40,870000,rank_nan,low_activity,high_entropy,5,1,-4,4,0.250000,6,0.186121,25.60,540,331,0.969351
94,358600,user_high,mid_activity,mid_entropy,6,2,-4,5,0.250000,6,0.142823,11.18,1804,874,0.951249
78,247700,ui_nan,mid_activity,low_entropy,3,0,-3,3,0.052632,9,0.272158,23.96,2339,745,0.907958
68,774700,ui_nan,mid_activity,low_entropy,5,2,-3,3,0.250000,6,0.179157,13.58,2106,577,0.909225
163,319200,user_low,high_activity,high_entropy,3,0,-3,3,0.428571,4,0.151270,9.10,3571,1806,0.967272
117,152000,item_high,mid_activity,high_entropy,2,0,-2,2,0.250000,6,0.161133,20.44,2286,1447,0.979585
152,358700,user_nan,high_activity,mid_entropy,2,0,-2,2,0.250000,6,0.179707,22.94,3063,1457,0.942734
18,14700,ui_nan,low_activity,low_entropy,4,2,-2,2,0.333333,5,0.135316,14.02,483,169,0.798679


### Поломка reranker по типам пользователей
Сравниваем влияние user, item, rank и user-item признаков для разных групп пользователей.

In [41]:
g2 = (d2
    .groupby(["act_group", "ent_group", "scenario"])
    .agg(
        users=("uid", "nunique"),
        delta_top10_rel=("delta_top10_rel", "mean"),
        relevant_dropped=("relevant_dropped_from_top10", "mean"),
        overlap_top10=("overlap_top10", "mean"),
        changed_items=("changed_items", "mean"),
        score_shift=("mean_abs_score_shift", "mean"),
        rank_shift=("mean_abs_rank_shift", "mean")
    )
    .reset_index()
    .sort_values(
        ["delta_top10_rel", "relevant_dropped", "changed_items"],
        ascending=[True, False, False]
    ))

g2.head(25)

C:\Users\vedma\AppData\Local\Temp\ipykernel_9052\3692714072.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["act_group", "ent_group", "scenario"])


,act_group,ent_group,scenario,users,delta_top10_rel,relevant_dropped,overlap_top10,changed_items,score_shift,rank_shift
5,low_activity,low_entropy,ui_default,2,-5.0,5.5,0.192982,7.0,0.146572,11.73
6,low_activity,low_entropy,ui_nan,2,-5.0,5.5,0.192982,7.0,0.146572,11.73
35,mid_activity,low_entropy,ui_default,2,-3.0,3.0,0.151316,7.5,0.225657,18.77
36,mid_activity,low_entropy,ui_nan,2,-3.0,3.0,0.151316,7.5,0.225657,18.77
47,mid_activity,mid_entropy,user_high,2,-2.0,3.0,0.180556,7.0,0.127189,11.68
23,low_activity,high_entropy,rank_nan,2,-2.0,2.0,0.213235,6.5,0.140075,21.97
25,low_activity,high_entropy,ui_default,2,-1.5,2.0,0.394231,4.5,0.026711,7.74
26,low_activity,high_entropy,ui_nan,2,-1.5,2.0,0.394231,4.5,0.026711,7.74
42,mid_activity,mid_entropy,item_nan,2,-1.5,2.0,0.394231,4.5,0.111883,11.15
88,high_activity,high_entropy,user_low,2,-1.5,1.5,0.623377,2.5,0.173405,10.23


### Пример изменения top-10 на втором этапе
Показываем конкретный кейс, где поломка признаков reranker резко переставляет выдачу.

In [43]:
def show2(uid, s):
    d = test[test["uid"] == uid].copy()
    b = br2(d, "base").copy()
    x = br2(d, s).copy()

    b["score"] = ranker.predict(b[cols])
    x["score"] = ranker.predict(x[cols])

    bt = b.sort_values("score", ascending=False).head(10)[[
        "item_id", "label", "elsa_rank", "score",
        "ui_n_events", "mean_played_ratio", "has_like",
        "days_before_test", "interaction_weight"
    ]].copy()

    xt = x.sort_values("score", ascending=False).head(10)[[
        "item_id", "label", "elsa_rank", "score",
        "ui_n_events", "mean_played_ratio", "has_like",
        "days_before_test", "interaction_weight"
    ]].copy()

    bt["base_rank"] = range(1, len(bt) + 1)
    xt["broken_rank"] = range(1, len(xt) + 1)

    z = bt.merge(xt, on="item_id", how="outer", suffixes=("_base", "_broken"))

    z["status"] = np.where(
        z["base_rank"].notna() & z["broken_rank"].notna(),
        "kept",
        np.where(z["base_rank"].notna(), "dropped_from_top10", "new_in_top10")
    )

    z["score_delta"] = z["score_broken"] - z["score_base"]
    z["rank_delta"] = z["broken_rank"] - z["base_rank"]

    return z.sort_values(["base_rank", "broken_rank"], na_position="last")

uid1 = int(w2.iloc[0]["uid"])
s1 = w2.iloc[0]["scenario"]

print(uid1, s1)
show2(uid1, s1)

268700 ui_nan


,item_id,label_base,elsa_rank_base,score_base,ui_n_events_base,mean_played_ratio_base,has_like_base,days_before_test_base,interaction_weight_base,base_rank,...,score_broken,ui_n_events_broken,mean_played_ratio_broken,has_like_broken,days_before_test_broken,interaction_weight_broken,broken_rank,status,score_delta,rank_delta
8,4119982,1.0,61.0,0.655556,16.0,75.812500,0.0,49.707465,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10,NaN,NaN
14,6372339,1.0,13.0,0.574778,10.0,10.800000,0.0,41.076678,0.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10,NaN,NaN
6,2368038,1.0,31.0,0.559624,9.0,39.333333,0.0,61.217303,0.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10,NaN,NaN
17,8334806,1.0,20.0,0.553577,8.0,62.500000,0.0,15.335938,0.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10,NaN,NaN
11,5865638,1.0,34.0,0.470969,7.0,51.857143,0.0,41.039062,0.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10,NaN,NaN
0,89163,1.0,80.0,0.457118,5.0,51.200000,0.0,10.898148,0.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10,NaN,NaN
10,5485941,1.0,69.0,0.425951,6.0,83.333333,0.0,19.731192,0.0,7.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10,NaN,NaN
3,1133883,1.0,52.0,0.417501,6.0,66.666667,0.0,41.063368,0.0,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10,NaN,NaN
5,1981007,1.0,92.0,0.405052,3.0,33.666667,0.0,19.758970,0.0,9.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10,NaN,NaN
18,8708659,0.0,6.0,0.364864,6.0,66.666667,0.0,96.466146,0.0,10.0,...,-0.259619,NaN,NaN,NaN,NaN,NaN,5.0,kept,-0.624483,-5.0


Самая опасная поломка на reranker-этапе — это user-item признаки. При ui_nan / ui_default в среднем меняется 6.33 айтема из top-10, overlap падает до 0.25, а из top-10 выпадает примерно 2 релевантных айтема. Это значит, что reranker сильнее всего зависит не просто от пользователя и не просто от популярности трека, а от конкретной связи пользователь-трек.

На конкретном пользователе 268700 видно совсем жестко: при ui_nan было 9 релевантных объектов в top-10, стало 1. То есть модель не просто чуть переставила выдачу, а фактически потеряла почти весь полезный top-10.

user_high, user_low, item_nan, item_low, item_high тоже меняют выдачу, но слабее: обычно меняется 2.6–2.9 айтема из 10. rank_nan и rank_reverse тоже влияют, но не так критично, как user-item блок.

In [46]:
def brf(d, c, m):
    x = d.copy()

    if m == "nan":
        x[c] = np.nan

    if m == "zero":
        x[c] = 0

    if m == "low":
        x[c] = x[c] * 0.1

    if m == "high":
        x[c] = x[c] * 2

    if m == "median":
        x[c] = x[c].median()

    return x

In [48]:
modes = ["nan", "zero", "low", "high", "median"]

r3 = []

for uid in uids:
    d = test[test["uid"] == uid].copy()

    for c in cols:
        for m in modes:
            x = brf(d, c, m)
            out = calc2(d, x, uid, c + "__" + m)
            out["feature"] = c
            out["mode"] = m

            if c in uc:
                out["block"] = "user"
            elif c in ic:
                out["block"] = "item"
            elif c in vc:
                out["block"] = "user_item"
            elif c == "elsa_rank":
                out["block"] = "rank"
            else:
                out["block"] = "other"

            r3.append(out)

s3 = pd.DataFrame(r3)
s3.head()

,uid,scenario,base_rel_pool,rel_pool,base_top10_rel,top10_rel,delta_top10_rel,relevant_dropped_from_top10,hit,overlap_top10,changed_items,mean_abs_score_shift,max_abs_score_shift,mean_abs_rank_shift,max_abs_rank_shift,feature,mode,block
0,268700,elsa_rank__nan,13,13,9,9,0,0,1,1.0,0,0.434506,0.696514,25.52,80,elsa_rank,nan,rank
1,268700,elsa_rank__zero,13,13,9,9,0,0,1,1.0,0,0.434506,0.696514,25.52,80,elsa_rank,zero,rank
2,268700,elsa_rank__low,13,13,9,9,0,0,1,1.0,0,0.314419,0.560902,18.30,75,elsa_rank,low,rank
3,268700,elsa_rank__high,13,13,9,9,0,0,1,1.0,0,0.035758,0.272212,6.80,42,elsa_rank,high,rank
4,268700,elsa_rank__median,13,13,9,9,0,0,1,1.0,0,0.065718,0.577008,16.22,60,elsa_rank,median,rank


### Поломка отдельных фичей
Проверяем, какие конкретные признаки сильнее всего меняют top-10 при пропуске, обнулении или искажении.

In [49]:
f3 = (s3
    .groupby(["block", "feature", "mode"])
    .agg(
        users=("uid", "nunique"),
        delta_top10_rel=("delta_top10_rel", "mean"),
        relevant_dropped=("relevant_dropped_from_top10", "mean"),
        overlap_top10=("overlap_top10", "mean"),
        changed_items=("changed_items", "mean"),
        score_shift=("mean_abs_score_shift", "mean"),
        rank_shift=("mean_abs_rank_shift", "mean")
    )
    .reset_index()
    .sort_values(
        ["overlap_top10", "relevant_dropped", "changed_items"],
        ascending=[True, False, False]
    ))

f3.head(30)

,block,feature,mode,users,delta_top10_rel,relevant_dropped,overlap_top10,changed_items,score_shift,rank_shift
133,user_item,ui_n_events,nan,18,-0.166667,1.055556,0.425368,4.333333,0.062338,3.923333
134,user_item,ui_n_events,zero,18,-0.166667,1.055556,0.425368,4.333333,0.062338,3.923333
131,user_item,ui_n_events,low,18,-0.111111,1.000000,0.425368,4.333333,0.060361,3.887778
132,user_item,ui_n_events,median,18,-0.166667,1.055556,0.434082,4.222222,0.061484,3.902222
123,user_item,mean_played_ratio,nan,18,-0.277778,0.611111,0.651978,2.277778,0.039856,4.976667
124,user_item,mean_played_ratio,zero,18,-0.222222,0.555556,0.658083,2.222222,0.054361,5.292222
121,user_item,mean_played_ratio,low,18,-0.166667,0.555556,0.660395,2.166667,0.031271,3.784444
106,user_item,days_before_test,low,18,-0.388889,0.500000,0.677313,2.222222,0.027797,2.277778
107,user_item,days_before_test,median,18,-0.555556,0.666667,0.679514,2.222222,0.035607,2.491111
108,user_item,days_before_test,nan,18,-0.555556,0.611111,0.686636,2.166667,0.033980,2.446667


### Итоговая чувствительность по фичам
Агрегируем влияние каждой фичи без разделения на тип поломки.

In [52]:
f4 = (s3
    .groupby(["block", "feature"])
    .agg(
        avg_delta_top10_rel=("delta_top10_rel", "mean"),
        avg_relevant_dropped=("relevant_dropped_from_top10", "mean"),
        avg_overlap_top10=("overlap_top10", "mean"),
        avg_changed_items=("changed_items", "mean"),
        avg_score_shift=("mean_abs_score_shift", "mean"),
        avg_rank_shift=("mean_abs_rank_shift", "mean")
    )
    .reset_index()
    .sort_values(
        ["avg_overlap_top10", "avg_relevant_dropped", "avg_changed_items"],
        ascending=[True, False, False]
    ))

f4.head(20)

,block,feature,avg_delta_top10_rel,avg_relevant_dropped,avg_overlap_top10,avg_changed_items,avg_score_shift,avg_rank_shift
26,user_item,ui_n_events,-0.088889,0.877778,0.490751,3.766667,0.061217,4.156667
21,user_item,days_before_test,-0.400000,0.544444,0.698012,2.088889,0.029626,2.234667
24,user_item,mean_played_ratio,-0.144444,0.422222,0.704033,1.888889,0.057275,5.487778
8,rank,elsa_rank,-0.222222,0.388889,0.760029,1.555556,0.133618,14.328222
2,item,item_like_rate,-0.088889,0.277778,0.810645,1.166667,0.027692,5.470000
6,item,matrix_popularity,-0.022222,0.133333,0.825501,1.144444,0.032273,7.104444
3,item,item_n_events,-0.088889,0.288889,0.828738,1.188889,0.029575,4.406444
25,user_item,organic_ratio_ui,0.100000,0.166667,0.831695,1.088889,0.011847,1.415556
10,user,entropy_norm,-0.066667,0.177778,0.841091,1.044444,0.075982,6.132667
13,user,n_items,-0.066667,0.188889,0.854975,0.933333,0.015961,1.622667


Наиболее критичная отдельная фича — ui_n_events. Если она не приходит, обнуляется или сильно занижается, top-10 меняется в среднем на 4.2–4.3 айтема, overlap падает до 0.42, а из top-10 выпадает около 1 релевантного объекта. Это значит, что reranker сильно опирается на количество взаимодействий конкретного пользователя с конкретным треком.

Вторая группа критичных фичей — days_before_test и mean_played_ratio. Они меняют меньше айтемов, но всё равно заметно двигают выдачу. Это признаки свежести и качества взаимодействия: насколько давно был контакт и насколько хорошо пользователь дослушивал трек.

elsa_rank тоже влияет, но слабее, чем ui_n_events: при поломке ранга меняется около 1.5 айтема из top-10. Значит, reranker не просто повторяет первый этап, а реально использует дополнительные признаки.

User-фичи и item-фичи в среднем менее опасны по отдельности. Например, user_like_rate, n_listens, item_n_likes почти не ломают top-10. Это значит, что модель больше зависит от конкретной пары пользователь-трек, чем от общей активности пользователя или общей популярности трека.

### Худшие точечные поломки
Выбираем сценарии, где конкретная фича дала максимальную потерю релевантных объектов в top-10.

In [55]:
worst_feat = (s3
    .sort_values(
        ["delta_top10_rel", "relevant_dropped_from_top10", "changed_items"],
        ascending=[True, False, False]
    )
    .head(20))

worst_feat[[
    "uid", "feature", "mode", "block",
    "base_top10_rel", "top10_rel", "delta_top10_rel",
    "relevant_dropped_from_top10",
    "overlap_top10", "changed_items",
    "mean_abs_score_shift", "mean_abs_rank_shift"
]]

,uid,feature,mode,block,base_top10_rel,top10_rel,delta_top10_rel,relevant_dropped_from_top10,overlap_top10,changed_items,mean_abs_score_shift,mean_abs_rank_shift
1335,358600,days_before_test,nan,user_item,6,2,-4,4,0.176471,7,0.063469,5.44
1336,358600,days_before_test,zero,user_item,6,2,-4,4,0.176471,7,0.063469,5.44
1339,358600,days_before_test,median,user_item,6,2,-4,4,0.176471,7,0.075372,6.30
540,870000,elsa_rank,nan,rank,5,1,-4,4,0.250000,6,0.186121,25.60
541,870000,elsa_rank,zero,rank,5,1,-4,4,0.250000,6,0.186121,25.60
645,870000,ui_n_events,nan,user_item,5,1,-4,4,0.428571,4,0.009993,0.96
646,870000,ui_n_events,zero,user_item,5,1,-4,4,0.428571,4,0.009993,0.96
647,870000,ui_n_events,low,user_item,5,1,-4,4,0.428571,4,0.009993,0.96
649,870000,ui_n_events,median,user_item,5,1,-4,4,0.428571,4,0.009993,0.96
1338,358600,days_before_test,high,user_item,6,3,-3,3,0.176471,7,0.047133,4.32


### Детальный пример поломки фичи
Смотрим, какие айтемы выпали из top-10 при поломке конкретной фичи.

In [57]:
def showf(uid, c, m):
    d = test[test["uid"] == uid].copy()
    b = d.copy()
    x = brf(d, c, m)

    b["score"] = ranker.predict(b[cols])
    x["score"] = ranker.predict(x[cols])

    keep = [
        "item_id", "label", "elsa_rank", "score",
        c, "ui_n_events", "mean_played_ratio",
        "days_before_test", "interaction_weight",
        "matrix_popularity", "item_like_rate"
    ]

    keep = list(dict.fromkeys([i for i in keep if i in b.columns]))

    bt = b.sort_values("score", ascending=False).head(10)[keep].copy()
    xt = x.sort_values("score", ascending=False).head(10)[keep].copy()

    bt["base_rank"] = range(1, len(bt) + 1)
    xt["broken_rank"] = range(1, len(xt) + 1)

    z = bt.merge(xt, on="item_id", how="outer", suffixes=("_base", "_broken"))

    z["status"] = np.where(
        z["base_rank"].notna() & z["broken_rank"].notna(),
        "kept",
        np.where(z["base_rank"].notna(), "dropped_from_top10", "new_in_top10")
    )

    z["score_delta"] = z["score_broken"] - z["score_base"]
    z["rank_delta"] = z["broken_rank"] - z["base_rank"]

    return z.sort_values(["base_rank", "broken_rank"], na_position="last")

ex = worst_feat.iloc[0]
uid2 = int(ex["uid"])
c2 = ex["feature"]
m2 = ex["mode"]

print(uid2, c2, m2)
showf(uid2, c2, m2)

358600 days_before_test nan


,item_id,label_base,elsa_rank_base,score_base,days_before_test_base,ui_n_events_base,mean_played_ratio_base,interaction_weight_base,matrix_popularity_base,item_like_rate_base,...,days_before_test_broken,ui_n_events_broken,mean_played_ratio_broken,interaction_weight_broken,matrix_popularity_broken,item_like_rate_broken,broken_rank,status,score_delta,rank_delta
14,6275528,1.0,39.0,0.331290,55.160012,5.0,80.20,0.0,621.0,0.012544,...,NaN,5.0,80.20,0.0,621.0,0.012544,8.0,kept,-0.280793,7.0
16,6484680,1.0,5.0,0.329711,58.491030,4.0,77.75,0.0,449.0,0.009138,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10,NaN,NaN
0,644501,1.0,4.0,0.318197,58.622106,4.0,82.50,0.0,484.0,0.006423,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10,NaN,NaN
8,4952290,1.0,20.0,0.288454,58.984375,4.0,65.00,0.0,484.0,0.008654,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10,NaN,NaN
6,3588069,0.0,25.0,0.287537,54.876157,4.0,79.25,0.0,101.0,0.006369,...,NaN,4.0,79.25,0.0,101.0,0.006369,6.0,kept,-0.227547,1.0
3,2654961,0.0,18.0,0.259334,54.863426,3.0,68.00,0.0,380.0,0.013713,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10,NaN,NaN
2,2118649,0.0,31.0,0.258277,50.924769,4.0,76.50,0.0,628.0,0.014553,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10,NaN,NaN
1,966876,1.0,92.0,0.257824,58.566840,4.0,76.00,0.0,160.0,0.014070,...,NaN,4.0,76.00,0.0,160.0,0.014070,4.0,kept,-0.167147,-4.0
13,6197362,0.0,16.0,0.251993,55.332465,3.0,83.00,0.0,254.0,0.006985,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10,NaN,NaN
5,3061013,1.0,45.0,0.243519,55.376157,4.0,75.50,0.0,667.0,0.014497,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dropped_from_top10,NaN,NaN


In [59]:
exs = worst_feat[["uid", "feature", "mode", "block"]].drop_duplicates().head(5)

tabs = []

for _, r in exs.iterrows():
    z = showf(int(r["uid"]), r["feature"], r["mode"])
    z["uid"] = int(r["uid"])
    z["feature"] = r["feature"]
    z["mode"] = r["mode"]
    tabs.append(z)

detail = pd.concat(tabs, ignore_index=True)

detail[[
    "uid", "feature", "mode", "item_id", "status",
    "label_base", "label_broken",
    "base_rank", "broken_rank",
    "score_base", "score_broken",
    "score_delta", "rank_delta"
]].head(60)

,uid,feature,mode,item_id,status,label_base,label_broken,base_rank,broken_rank,score_base,score_broken,score_delta,rank_delta
0,358600,days_before_test,nan,6275528,kept,1.0,1.0,1.0,8.0,0.331290,0.050497,-0.280793,7.0
1,358600,days_before_test,nan,6484680,dropped_from_top10,1.0,NaN,2.0,NaN,0.329711,NaN,NaN,NaN
2,358600,days_before_test,nan,644501,dropped_from_top10,1.0,NaN,3.0,NaN,0.318197,NaN,NaN,NaN
3,358600,days_before_test,nan,4952290,dropped_from_top10,1.0,NaN,4.0,NaN,0.288454,NaN,NaN,NaN
4,358600,days_before_test,nan,3588069,kept,0.0,0.0,5.0,6.0,0.287537,0.059990,-0.227547,1.0
5,358600,days_before_test,nan,2654961,dropped_from_top10,0.0,NaN,6.0,NaN,0.259334,NaN,NaN,NaN
6,358600,days_before_test,nan,2118649,dropped_from_top10,0.0,NaN,7.0,NaN,0.258277,NaN,NaN,NaN
7,358600,days_before_test,nan,966876,kept,1.0,1.0,8.0,4.0,0.257824,0.090677,-0.167147,-4.0
8,358600,days_before_test,nan,6197362,dropped_from_top10,0.0,NaN,9.0,NaN,0.251993,NaN,NaN,NaN
9,358600,days_before_test,nan,3061013,dropped_from_top10,1.0,NaN,10.0,NaN,0.243519,NaN,NaN,NaN


Самая опасная отдельная фича в среднем — ui_n_events. Если она ломается, top-10 меняется примерно на 3.77 айтема, overlap падает до 0.49. Это значит, что модель сильно завязана на количество взаимодействий конкретного пользователя с конкретным треком.

Но в худших индивидуальных кейсах сильнее всего стреляет days_before_test. Для пользователя 358600 при поломке days_before_test было 6 релевантных объектов в top-10, стало 2. Из top-10 выпало 4 релевантных объекта, поменялось 7 айтемов. Это уже хороший пример для бизнеса: если ломается фича свежести взаимодействия, модель начинает неправильно понимать, какие треки актуальны для пользователя сейчас.

Важно: elsa_rank тоже влияет, но слабее, чем user-item фичи. Значит, reranker не просто повторяет порядок первого этапа, а реально использует собранные признаки.

Теперь надо закрыть важную часть ТЗ: “убираешь пользователей из обучения”. Сейчас мы проверяли поломки на выбранных пользователях, но ranker был уже готовый. Дальше делаем корректнее: обучим отдельный LightGBM без выбранных 18 пользователей и сравним поведение. Это сильнее соответствует формулировке задания.

### Проверка train/test для выбранных пользователей
Проверяем, участвовали ли выбранные пользователи в обучении модели.

In [62]:
train_uids = set(train["uid"])
sel_uids = set(uids)

pd.DataFrame({
    "metric": [
        "selected users",
        "selected users in train",
        "selected users in test"
    ],
    "value": [
        len(sel_uids),
        len(sel_uids & train_uids),
        test[test["uid"].isin(sel_uids)]["uid"].nunique()
    ]
})

,metric,value
0,selected users,18
1,selected users in train,0
2,selected users in test,18


In [64]:
tr2 = train[~train["uid"].isin(uids)].copy()
tr2 = tr2.sort_values("uid")

grp = tr2.groupby("uid").size().values
ds = lgb.Dataset(tr2[cols], label=tr2["label"], group=grp)

par = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [10],
    "learning_rate": 0.05,
    "num_leaves": 51,
    "min_data_in_leaf": 50,
    "verbose": -1,
    "seed": 42
}

ranker_oos = lgb.train(par, ds, num_boost_round=80)

### Проверка устойчивости после переобучения
Сравниваем исходный reranker и модель, переобученную без выбранных пользователей.

In [66]:
def calc_oos(uid):
    d = test[test["uid"] == uid].copy()

    d["score_old"] = ranker.predict(d[cols])
    d["score_oos"] = ranker_oos.predict(d[cols])

    a = d.sort_values("score_old", ascending=False).head(10)
    b = d.sort_values("score_oos", ascending=False).head(10)

    ai = set(a["item_id"])
    bi = set(b["item_id"])

    return {
        "uid": uid,
        "old_top10_rel": a["label"].sum(),
        "oos_top10_rel": b["label"].sum(),
        "delta_top10_rel": b["label"].sum() - a["label"].sum(),
        "old_hit": int(a["label"].sum() > 0),
        "oos_hit": int(b["label"].sum() > 0),
        "overlap_top10": len(ai & bi) / max(len(ai | bi), 1),
        "changed_items": 10 - len(ai & bi),
        "old_avg_score": a["score_old"].mean(),
        "oos_avg_score": b["score_oos"].mean()
    }

oos = pd.DataFrame([calc_oos(uid) for uid in uids])

oos = oos.merge(
    sel[[
        "uid", "act_group", "ent_group", "user_n_events",
        "n_unique_items", "entropy_norm", "rel_cand"
    ]],
    on="uid",
    how="left"
)

oos.sort_values(["overlap_top10", "changed_items"], ascending=[True, False])

,uid,old_top10_rel,oos_top10_rel,delta_top10_rel,old_hit,oos_hit,overlap_top10,changed_items,old_avg_score,oos_avg_score,act_group,ent_group,user_n_events,n_unique_items,entropy_norm,rel_cand
11,152000,2,0,-2,1,0,0.111111,8,-0.020631,-0.019284,mid_activity,high_entropy,2286,1447,0.979585,7.0
5,865000,0,0,0,0,0,0.333333,5,-0.013355,-0.150476,low_activity,high_entropy,748,534,0.972238,12.0
14,394900,1,3,2,1,1,0.333333,5,0.268329,0.685233,high_activity,mid_entropy,2639,862,0.960095,14.0
15,358700,2,3,1,1,1,0.428571,4,-0.022454,-0.184422,high_activity,mid_entropy,3063,1457,0.942734,10.0
16,319200,3,2,-1,1,1,0.428571,4,0.241682,0.715481,high_activity,high_entropy,3571,1806,0.967272,20.0
1,14700,4,4,0,1,1,0.538462,3,0.268159,0.531789,low_activity,low_entropy,483,169,0.798679,10.0
3,790600,1,1,0,1,1,0.538462,3,0.180167,0.408366,low_activity,mid_entropy,690,366,0.963046,6.0
4,870000,5,5,0,1,1,0.538462,3,0.130292,0.329021,low_activity,high_entropy,540,331,0.969351,13.0
8,58300,2,1,-1,1,1,0.538462,3,0.148442,0.387774,mid_activity,mid_entropy,2327,1225,0.961728,14.0
13,155300,6,6,0,1,1,0.538462,3,0.237937,0.442832,high_activity,low_entropy,2950,714,0.874182,12.0


Выбранные пользователи отсутствуют в train, но присутствуют в test. Значит, анализ проводится на пользователях, которых reranker не видел при обучении. Это корректно для проверки поведения модели на отложенных пользователях.

Сравнение исходного ranker и переобученного ranker показывает, что модель в целом сохраняет hit для большинства пользователей, но состав top-10 может заметно отличаться: у части пользователей меняется 5–8 айтемов из 10. Это означает, что итоговая выдача чувствительна не только к поломке признаков, но и к параметрам/версии reranker. Поэтому для бизнеса важно мониторить не только метрики качества, но и стабильность top-10 после переобучения.